In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.ecommerce.silver_events")

gold = (
    silver
    .groupBy("product_id")
    .agg(
        F.count(F.when(F.col("event_type")=="view", True)).alias("views"),
        F.count(F.when(F.col("event_type")=="cart", True)).alias("cart_adds"),
        F.count(F.when(F.col("event_type")=="purchase", True)).alias("purchases"),
        F.sum(F.when(F.col("event_type")=="purchase", F.col("price"))).alias("revenue")
    )
)

gold.show(5)


+----------+-----+---------+---------+------------------+
|product_id|views|cart_adds|purchases|           revenue|
+----------+-----+---------+---------+------------------+
|   1005159|57475|     1579|     1063|238752.03000000006|
|   5701087| 4819|        0|       80| 4118.400000000001|
|  38300497|   14|        0|        0|              NULL|
|  41100005|  458|        0|        1|             68.21|
|  14700174|  131|        0|        0|              NULL|
+----------+-----+---------+---------+------------------+
only showing top 5 rows


In [0]:
gold_path = "/Volumes/workspace/ecommerce/gold/product_performance"

gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_path)

print("Gold Delta data written successfully")


Gold Delta data written successfully


In [0]:
%sql
DROP TABLE IF EXISTS workspace.ecommerce.gold_product_performance;


In [0]:
%sql
CREATE TABLE workspace.ecommerce.gold_product_performance
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/workspace/ecommerce/gold/product_performance`;


num_affected_rows,num_inserted_rows


In [0]:
df = spark.table("workspace.ecommerce.gold_product_performance").toPandas()
df.head()


,product_id,views,cart_adds,purchases,revenue
0,1005159,57475,1579,1063,238752.03
1,5701087,4819,0,80,4118.40
2,38300497,14,0,0,NaN
3,41100005,458,0,1,68.21
4,14700174,131,0,0,NaN


In [0]:
X = df[["views", "cart_adds"]]
y = df["purchases"]


In [0]:
import pandas as pd

df = spark.table("workspace.ecommerce.gold_product_performance").toPandas()
df.head()


,product_id,views,cart_adds,purchases,revenue
0,1005159,57475,1579,1063,238752.03
1,5701087,4819,0,80,4118.40
2,38300497,14,0,0,NaN
3,41100005,458,0,1,68.21
4,14700174,131,0,0,NaN


In [0]:
X = df[["views", "cart_adds"]]
y = df["purchases"]


In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [0]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression

with mlflow.start_run(run_name="linear_regression_v1"):

    # Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)

    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Evaluate
    r2 = model.score(X_test, y_test)
    mlflow.log_metric("r2_score", r2)

    # Log model
    mlflow.sklearn.log_model(model, "model")

print(f"R² Score: {r2:.4f}")


2026/01/20 13:17:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


R² Score: 0.9817


In [0]:
with mlflow.start_run(run_name="linear_regression_v2"):
    model = LinearRegression()
    model.fit(X_train, y_train)
    r2 = model.score(X_test, y_test)
    mlflow.log_metric("r2_score", r2)
